# q1: GSM8K with Qwen2.5-0.5B — 4 prompting methods
Run all cells from top to bottom on a Colab T4 GPU. Section 9 runs on dev (train 1-10), section 11 switches to test (test 1-30), and section 12 prints the final table.

**Notes (decided on dev):**
- Max 512 new tokens, because with 300 some answers were cut off.
- The answer is read from "The answer is X", then `\boxed{X}`, otherwise from the last sentence.
- Self-consistency: 4 samples (temperature 0.7) generated in one batched call, still counted as 4 calls.
- Reflection: the review call does not include the 4 demos, because otherwise the model sometimes checked a demo instead of its own answer.


## 0. Install (only needed on Colab)

In [1]:
!pip install -q transformers datasets accelerate torch

## 1. Imports

In [2]:
import json
import re
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

## 2. Load model

In [3]:
name = "Qwen/Qwen2.5-0.5B-Instruct"

# pick the best available device automatically
if torch.cuda.is_available():
    device, dtype = "cuda", torch.float16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.float32
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).to(device)

Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 3. Load data (dev first; section 11 switches to test)

In [4]:
ds = load_dataset("openai/gsm8k", "main")
train, test = ds["train"], ds["test"]

MODE = "dev"
questions = [train[i] for i in range(0, 10)]      # train 1-10
print(MODE, len(questions), "questions")


def gold(example):
    """The correct number is the text after '####' (only used for scoring, never shown to the model)."""
    return example["answer"].split("####")[-1].strip().replace(",", "")


def clean_solution(example):
    """Turn a GSM8K solution into text the model can copy as a demo."""
    steps, final = example["answer"].split("####")
    steps = re.sub(r"<<.*?>>", "", steps).strip()     # remove <<48/2=24>> notes
    return f"{steps}\nThe answer is {final.strip()}."


for i, ex in enumerate(questions, 1):
    print(f"{i}. {ex['question']}\n   gold = {gold(ex)}\n")

dev 10 questions
1. Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
   gold = 72

2. Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
   gold = 10

3. Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
   gold = 5

4. Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?
   gold = 42

5. James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?
   gold = 624

6. Mark has a garden with flowers. He plant

## 4. Prompts (tuned on dev only)

In [5]:
SYSTEM = ("Solve the math problem step by step. "
          "End with: The answer is <number>.")

demos = []                                            # train 11-14
for i in range(10, 14):
    demos.append({"role": "user", "content": train[i]["question"]})
    demos.append({"role": "assistant", "content": clean_solution(train[i])})

REFLECT = ("Check your solution carefully for mistakes. If you find one, fix it. "
           "End with: The answer is <number>.")

## 5. Answer parsing

In [6]:
NUM = r"-?\d+(?:\.\d+)?"


def canon(x):
    """'18.00' -> '18', so equal numbers are equal strings (matters for majority voting)."""
    v = float(x)
    return str(int(v)) if v.is_integer() else str(round(v, 4))


def get_number(text):
    """Final answer: 'The answer is X', else \\boxed{X}, else from the last line that has a number:
    the first number after its last '=' / 'is', else its first number ('earned $10 for 50 minutes' -> 10)."""
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text).replace("$", "")      # 1,080 -> 1080, $5 -> 5
    m = (re.findall(r"answer is[\s:*\\(]*(" + NUM + ")", text, re.I)
         or re.findall(r"\\boxed\{\D{0,10}?(" + NUM + ")", text))
    if m:
        return canon(m[-1])
    lines = [re.sub(r"^\s*\d+[.)]\s", "", l) for l in text.splitlines()]   # drop "3. " list numbers
    lines = [l for l in lines if re.search(r"\d", l)]
    if not lines:
        return None
    after_eq = lines[-1].split("=")[-1]
    after_is = re.split(r"\b(?:is|are|was|were)\b", after_eq)[-1]
    for part in (after_is, after_eq, lines[-1]):
        nums = re.findall(NUM, part)
        if nums:
            return canon(nums[0])


def same(a, b):
    try:
        return a is not None and abs(float(a) - float(b)) < 1e-6
    except (TypeError, ValueError):
        return False

## 6. Model call

In [7]:
MAX_NEW_TOKENS = 512   # 300 cut off 2/10 zero-shot dev answers (see the notes at the top)

eos = model.generation_config.eos_token_id
END_IDS = set(eos if isinstance(eos, list) else [eos]) | {tokenizer.eos_token_id, tokenizer.pad_token_id}

tokens_used = 0        # total generated tokens
calls_made = 0         # total model calls (one per generated solution)
texts = []             # (output, n_tokens) of the current question, saved by the runner


def generate_texts(messages, **settings):
    """One generate() call. Every returned solution counts as one model call."""
    global tokens_used, calls_made
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
    ).to(device)
    n = inputs["input_ids"].shape[-1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, **settings)
    outputs = []
    for seq in out[:, n:].tolist():
        k = next((i + 1 for i, t in enumerate(seq) if t in END_IDS), len(seq))   # up to the end token
        text = tokenizer.decode(seq[:k], skip_special_tokens=True)
        tokens_used += k
        calls_made += 1
        texts.append((text, k))
        outputs.append(text)
    return outputs


def ask(messages):
    """One greedy solution."""
    return generate_texts(messages, do_sample=False, temperature=None, top_p=None, top_k=None)[0]


def sample(messages, k):
    """k independently sampled solutions, generated together in one batched call (= k model calls)."""
    return generate_texts(messages, do_sample=True, temperature=0.7, top_p=0.95, num_return_sequences=k)

## 7. The four methods

In [8]:
def zero_shot(q):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    return get_number(ask(msgs))


def four_shot(q):
    msgs = [{"role": "system", "content": SYSTEM}] + demos + [{"role": "user", "content": q}]
    return get_number(ask(msgs))


def self_consistency(q):
    msgs = [{"role": "system", "content": SYSTEM}] + demos + [{"role": "user", "content": q}]
    answers = [a for a in map(get_number, sample(msgs, 4)) if a is not None]
    return Counter(answers).most_common(1)[0][0] if answers else None     # tie -> earliest sample


def reflection(q):
    msgs = [{"role": "system", "content": SYSTEM}] + demos + [{"role": "user", "content": q}]
    first = ask(msgs)                                    # the same solution four-shot would give
    review = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q},   # no demos here, see notes
              {"role": "assistant", "content": first}, {"role": "user", "content": REFLECT}]
    return get_number(ask(review)) or get_number(first)  # keep the first answer if the revision has no number


methods = {"zero-shot": zero_shot, "four-shot": four_shot,
           "self-consistency": self_consistency, "reflection": reflection}

## 8. Runner + `show`
`run_method` saves the model's full output(s) for every question.
`show` only prints those saved outputs next to the reference solution — it does **not** run the model again.

In [9]:
results = {}   # method name -> (correct, tokens, calls)
logs = {}      # (mode, method name) -> [(example, pred, ok, [(output, n_tokens), ...])]


def run_method(mname):
    global tokens_used, calls_made
    torch.manual_seed(0)
    tokens_used, calls_made = 0, 0
    logs[(MODE, mname)] = []
    for ex in questions:
        texts.clear()
        pred = methods[mname](ex["question"])
        ok = same(pred, gold(ex))
        logs[(MODE, mname)].append((ex, pred, ok, list(texts)))
        print(f"  [{mname}] pred={pred} gold={gold(ex)} {'OK' if ok else 'WRONG'}")
    correct = sum(r[2] for r in logs[(MODE, mname)])
    cut = sum(k >= MAX_NEW_TOKENS for r in logs[(MODE, mname)] for _, k in r[3])
    results[mname] = (correct, tokens_used, calls_made)
    print(f"{mname}: {correct}/{len(questions)}  tokens={tokens_used}  calls={calls_made}  "
          f"(outputs cut off at {MAX_NEW_TOKENS} tokens: {cut})")


def show(mname):
    """Print what run_method saved: each question, the model's full output(s) and the reference solution."""
    for i, (ex, pred, ok, outs) in enumerate(logs[(MODE, mname)], 1):
        print(f"========== Q{i}  {'OK' if ok else 'WRONG'}  pred={pred}  gold={gold(ex)} ==========")
        print("QUESTION:", ex["question"])
        for j, (t, k) in enumerate(outs, 1):
            print(f"\n--- model output {j} ({k} tokens{', CUT OFF' if k >= MAX_NEW_TOKENS else ''}) ---\n{t}")
        print(f"\n--- reference solution ---\n{clean_solution(ex)}\n")


def print_table(title, res, n):
    print(f"\n{title}")
    print(f"{'method':<18}{'accuracy':<14}{'tokens':<10}{'calls':<8}")
    for mname, (c, t, k) in res.items():
        print(f"{mname:<18}{f'{c}/{n} ({100 * c / n:.0f}%)':<14}{t:<10}{k:<8}")

## 9. Dev run (train 1-10)

### zero-shot

In [10]:
run_method("zero-shot")

  [zero-shot] pred=72 gold=72 OK
  [zero-shot] pred=10 gold=10 OK
  [zero-shot] pred=55 gold=5 WRONG
  [zero-shot] pred=42 gold=42 OK
  [zero-shot] pred=2184 gold=624 WRONG
  [zero-shot] pred=35 gold=35 OK
  [zero-shot] pred=40 gold=48 WRONG
  [zero-shot] pred=16 gold=16 OK
  [zero-shot] pred=184 gold=41 WRONG
  [zero-shot] pred=1080 gold=990 WRONG
zero-shot: 5/10  tokens=2321  calls=10  (outputs cut off at 512 tokens: 0)


In [11]:
show("zero-shot")

========== Q1  OK  pred=72  gold=72 ==========
QUESTION: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

--- model output 1 (213 tokens) ---
To determine the total number of clips Natalia sold in April and May, we need to follow these steps:

1. Identify the number of clips sold in April.
2. Calculate the number of clips sold in May based on the information given.
3. Add the number of clips sold in April and May together.

Step 1: Identify the number of clips sold in April.
Natalia sold clips to 48 of her friends in April.

Step 2: Calculate the number of clips sold in May.
In May, Natalia sold half as many clips as she did in April. Therefore, the number of clips sold in May is:
\[ \frac{48}{2} = 24 \]

Step 3: Add the number of clips sold in April and May together.
The total number of clips sold in April and May is:
\[ 48 + 24 = 72 \]

Thus, the total number of clips Natalia s

### four-shot

In [12]:
run_method("four-shot")

  [four-shot] pred=72 gold=72 OK
  [four-shot] pred=40.8 gold=10 WRONG
  [four-shot] pred=40 gold=5 WRONG
  [four-shot] pred=42 gold=42 OK
  [four-shot] pred=624 gold=624 OK
  [four-shot] pred=109 gold=35 WRONG
  [four-shot] pred=48 gold=48 OK
  [four-shot] pred=16 gold=16 OK
  [four-shot] pred=117 gold=41 WRONG
  [four-shot] pred=756 gold=990 WRONG
four-shot: 5/10  tokens=1211  calls=10  (outputs cut off at 512 tokens: 0)


In [13]:
show("four-shot")

========== Q1  OK  pred=72  gold=72 ==========
QUESTION: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

--- model output 1 (72 tokens) ---
In April, Natalia sold clips to 48 friends.
In May, she sold half as many clips as in April, which means she sold 48 / 2 = 24 clips.
Altogether, Natalia sold 48 + 24 = 72 clips.

Therefore, the answer is 72.

--- reference solution ---
Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.
The answer is 72.

========== Q2  WRONG  pred=40.8  gold=10 ==========
QUESTION: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

--- model output 1 (196 tokens) ---
To calculate Weng's earnings, we need to convert the time she worked into hours. There are 60 minutes in an hour, so 50 minutes is equal to 50/60 hours.

Earnings = (Time worked in hours)

### self-consistency

In [14]:
run_method("self-consistency")

  [self-consistency] pred=72 gold=72 OK
  [self-consistency] pred=10 gold=10 OK
  [self-consistency] pred=5 gold=5 OK
  [self-consistency] pred=42 gold=42 OK
  [self-consistency] pred=312 gold=624 WRONG
  [self-consistency] pred=58.5 gold=35 WRONG
  [self-consistency] pred=48 gold=48 OK
  [self-consistency] pred=8 gold=16 WRONG
  [self-consistency] pred=57 gold=41 WRONG
  [self-consistency] pred=720 gold=990 WRONG
self-consistency: 5/10  tokens=4591  calls=40  (outputs cut off at 512 tokens: 0)


In [15]:
show("self-consistency")

========== Q1  OK  pred=72  gold=72 ==========
QUESTION: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

--- model output 1 (78 tokens) ---
In April, Natalia sold 48 clips.
In May, she sold half as many clips as in April, which is 48 / 2 = 24 clips.

Therefore, Natalia sold a total of 48 (April) + 24 (May) = 72 clips altogether in April and May.

The answer is 72.

--- model output 2 (81 tokens) ---
In April, Natalia sold clips to 48 friends.
In May, she sold half as many clips as in April, which is 48 / 2 = 24 clips.
To find the total number of clips sold in April and May, we add the two numbers together: 48 + 24 = 72 clips.
The answer is 72.

--- model output 3 (99 tokens) ---
In May, Natalia sold half as many clips as she did in April, which was 48 clips.
So, in May, Natalia sold 48 / 2 = 24 clips.

To find out how many clips Natalia sold in both April and May, we add the two

### reflection

In [16]:
run_method("reflection")

  [reflection] pred=72 gold=72 OK
  [reflection] pred=10 gold=10 OK
  [reflection] pred=5 gold=5 OK
  [reflection] pred=42 gold=42 OK
  [reflection] pred=624 gold=624 OK
  [reflection] pred=35 gold=35 OK
  [reflection] pred=48 gold=48 OK
  [reflection] pred=16 gold=16 OK
  [reflection] pred=0 gold=41 WRONG
  [reflection] pred=1035 gold=990 WRONG
reflection: 8/10  tokens=3349  calls=20  (outputs cut off at 512 tokens: 0)


In [17]:
show("reflection")

========== Q1  OK  pred=72  gold=72 ==========
QUESTION: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

--- model output 1 (72 tokens) ---
In April, Natalia sold clips to 48 friends.
In May, she sold half as many clips as in April, which means she sold 48 / 2 = 24 clips.
Altogether, Natalia sold 48 + 24 = 72 clips.

Therefore, the answer is 72.

--- model output 2 (121 tokens) ---
I apologize for the mistake in my previous response. Let's solve this step-by-step again:

1. In April, Natalia sold clips to 48 friends.
2. In May, she sold half as many clips as in April, so she sold 48 / 2 = 24 clips.
3. To find out how many clips Natalia sold altogether in April and May, we add the number of clips sold in each month:
   48 (April) + 24 (May) = 72

Therefore, the answer is 72.

--- reference solution ---
Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether 

## 10. Dev summary table

In [18]:
print_table("DEV (train 1-10)", results, len(questions))


DEV (train 1-10)
method            accuracy      tokens    calls   
zero-shot         5/10 (50%)    2321      10      
four-shot         5/10 (50%)    1211      10      
self-consistency  5/10 (50%)    4591      40      
reflection        8/10 (80%)    3349      20      


## 11. Final test run (test 1-30, prompts fixed on dev)
Run the switch cell once. Do **not** re-run sections 3 or 8 after this, or everything goes back to dev.

In [19]:
if MODE == "dev":                                # safe to re-run: only switches once
    dev_results, results = dict(results), {}
    MODE = "test"
    questions = [test[i] for i in range(0, 30)]  # test 1-30
print(MODE, len(questions), "questions")

test 30 questions


### zero-shot (test)

In [20]:
run_method("zero-shot")

  [zero-shot] pred=18 gold=18 OK
  [zero-shot] pred=3 gold=3 OK
  [zero-shot] pred=-65000 gold=70000 WRONG
  [zero-shot] pred=540 gold=540 OK
  [zero-shot] pred=420 gold=20 WRONG
  [zero-shot] pred=48 gold=64 WRONG
  [zero-shot] pred=260 gold=260 OK
  [zero-shot] pred=50 gold=160 WRONG
  [zero-shot] pred=620 gold=45 WRONG
  [zero-shot] pred=1390 gold=460 WRONG
  [zero-shot] pred=366 gold=366 OK
  [zero-shot] pred=694 gold=694 OK
  [zero-shot] pred=16.5 gold=13 WRONG
  [zero-shot] pred=9 gold=18 WRONG
  [zero-shot] pred=60 gold=60 OK
  [zero-shot] pred=4789 gold=125 WRONG
  [zero-shot] pred=230 gold=230 OK
  [zero-shot] pred=6020 gold=57500 WRONG
  [zero-shot] pred=7 gold=7 OK
  [zero-shot] pred=4 gold=6 WRONG
  [zero-shot] pred=17.33 gold=15 WRONG
  [zero-shot] pred=6 gold=14 WRONG
  [zero-shot] pred=7 gold=7 OK
  [zero-shot] pred=0.92 gold=8 WRONG
  [zero-shot] pred=14.625 gold=26 WRONG
  [zero-shot] pred=1 gold=2 WRONG
  [zero-shot] pred=233 gold=243 WRONG
  [zero-shot] pred=240 gold

In [21]:
show("zero-shot")

========== Q1  OK  pred=18  gold=18 ==========
QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

--- model output 1 (259 tokens) ---
To determine how much money Janet makes every day from selling the remaining eggs, we need to follow these steps:

1. Calculate the total number of eggs laid by the ducks each day.
2. Subtract the number of eggs eaten for breakfast and baked muffins from the total number of eggs.
3. Determine how many eggs are sold at the farmers' market each day.

Step 1: Calculate the total number of eggs laid by the ducks each day.
Janet's ducks lay 16 eggs per day.

Step 2: Subtract the number of eggs eaten for breakfast and baked muffins from the total number of eggs.
She eats 3 eggs for breakfast and bakes 4 muffins, so she eats

### four-shot (test)

In [22]:
run_method("four-shot")

  [four-shot] pred=18 gold=18 OK
  [four-shot] pred=3 gold=3 OK
  [four-shot] pred=65000 gold=70000 WRONG
  [four-shot] pred=180 gold=540 WRONG
  [four-shot] pred=2 gold=20 WRONG
  [four-shot] pred=24 gold=64 WRONG
  [four-shot] pred=260 gold=260 OK
  [four-shot] pred=33733.33 gold=160 WRONG
  [four-shot] pred=400 gold=45 WRONG
  [four-shot] pred=452 gold=460 WRONG
  [four-shot] pred=394 gold=366 WRONG
  [four-shot] pred=694 gold=694 OK
  [four-shot] pred=1 gold=13 WRONG
  [four-shot] pred=10.5 gold=18 WRONG
  [four-shot] pred=80 gold=60 WRONG
  [four-shot] pred=-4779 gold=125 WRONG
  [four-shot] pred=300 gold=230 WRONG
  [four-shot] pred=1150 gold=57500 WRONG
  [four-shot] pred=7 gold=7 OK
  [four-shot] pred=6 gold=6 OK
  [four-shot] pred=19 gold=15 WRONG
  [four-shot] pred=17 gold=14 WRONG
  [four-shot] pred=8 gold=7 WRONG
  [four-shot] pred=2 gold=8 WRONG
  [four-shot] pred=26 gold=26 OK
  [four-shot] pred=3 gold=2 WRONG
  [four-shot] pred=243 gold=243 OK
  [four-shot] pred=16 gold=

In [23]:
show("four-shot")

========== Q1  OK  pred=18  gold=18 ==========
QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

--- model output 1 (57 tokens) ---
She eats 3 + 4 = 7 eggs daily.
That means she sells 16 - 7 = 9 eggs daily.

At $2 per egg, she makes 9 * $2 = $18 every day.

The answer is 18.

--- reference solution ---
Janet sells 16 - 3 - 4 = 9 duck eggs a day.
She makes 9 * 2 = $18 every day at the farmer’s market.
The answer is 18.

========== Q2  OK  pred=3  gold=3 ==========
QUESTION: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

--- model output 1 (57 tokens) ---
It takes 2 bolts of blue fiber.
It takes half that amount of white fiber, which is 2 / 2 = 1 bolt of white fiber.
In total, it takes 2 + 1

### self-consistency (test)

In [24]:
run_method("self-consistency")

  [self-consistency] pred=18 gold=18 OK
  [self-consistency] pred=3 gold=3 OK
  [self-consistency] pred=120000 gold=70000 WRONG
  [self-consistency] pred=180 gold=540 WRONG
  [self-consistency] pred=40 gold=20 WRONG
  [self-consistency] pred=28.8 gold=64 WRONG
  [self-consistency] pred=260 gold=260 OK
  [self-consistency] pred=8 gold=160 WRONG
  [self-consistency] pred=670 gold=45 WRONG
  [self-consistency] pred=400 gold=460 WRONG
  [self-consistency] pred=366 gold=366 OK
  [self-consistency] pred=694 gold=694 OK
  [self-consistency] pred=29 gold=13 WRONG
  [self-consistency] pred=5 gold=18 WRONG
  [self-consistency] pred=60 gold=60 OK
  [self-consistency] pred=96 gold=125 WRONG
  [self-consistency] pred=10.16 gold=230 WRONG
  [self-consistency] pred=1150 gold=57500 WRONG
  [self-consistency] pred=7 gold=7 OK
  [self-consistency] pred=1 gold=6 WRONG
  [self-consistency] pred=17 gold=15 WRONG
  [self-consistency] pred=23 gold=14 WRONG
  [self-consistency] pred=8 gold=7 WRONG
  [self-con

In [25]:
show("self-consistency")

========== Q1  OK  pred=18  gold=18 ==========
QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

--- model output 1 (67 tokens) ---
She eats 3 + 4 = 7 eggs every day.
That means Janet makes 16 - 7 = 9 eggs every day.

Each egg is sold for $2, so she makes 9 x $2 = $18 every day at the farmers' market.
The answer is 18.

--- model output 2 (56 tokens) ---
She makes 16-3=13 eggs a day.
She makes 13*2=$26 selling eggs.
That means she makes 13+26=$39 at the farmers' market.

Therefore, the answer is 39.

--- model output 3 (117 tokens) ---
She has 16 x 3 = 48 breakfast eggs left for herself. She has 16 x 4 = 64 baked muffins. That means she has 16 - 48 - 64 = -112 eggs available for sale. But that can't happen because there are no eggs left to sell. So

### reflection (test)

In [26]:
run_method("reflection")

  [reflection] pred=18 gold=18 OK
  [reflection] pred=3 gold=3 OK
  [reflection] pred=170000 gold=70000 WRONG
  [reflection] pred=180 gold=540 WRONG
  [reflection] pred=0 gold=20 WRONG
  [reflection] pred=24 gold=64 WRONG
  [reflection] pred=260 gold=260 OK
  [reflection] pred=140 gold=160 WRONG
  [reflection] pred=340 gold=45 WRONG
  [reflection] pred=880 gold=460 WRONG
  [reflection] pred=366 gold=366 OK
  [reflection] pred=694 gold=694 OK
  [reflection] pred=1 gold=13 WRONG
  [reflection] pred=15 gold=18 WRONG
  [reflection] pred=60 gold=60 OK
  [reflection] pred=5125 gold=125 WRONG
  [reflection] pred=0 gold=230 WRONG
  [reflection] pred=1150 gold=57500 WRONG
  [reflection] pred=7 gold=7 OK
  [reflection] pred=4 gold=6 WRONG
  [reflection] pred=9 gold=15 WRONG
  [reflection] pred=17 gold=14 WRONG
  [reflection] pred=7 gold=7 OK
  [reflection] pred=2 gold=8 WRONG
  [reflection] pred=26 gold=26 OK
  [reflection] pred=2 gold=2 OK
  [reflection] pred=233 gold=243 WRONG
  [reflection] p

In [27]:
show("reflection")

========== Q1  OK  pred=18  gold=18 ==========
QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

--- model output 1 (57 tokens) ---
She eats 3 + 4 = 7 eggs daily.
That means she sells 16 - 7 = 9 eggs daily.

At $2 per egg, she makes 9 * $2 = $18 every day.

The answer is 18.

--- model output 2 (255 tokens) ---
I apologize for the mistake in my previous response. Let's solve this step-by-step again:

1. Janet's ducks lay 16 eggs per day.
2. She eats 3 eggs for breakfast every morning.
3. She bakes muffins for her friends every day with 4 muffins.
4. She sells the remaining eggs at the farmers' market daily for $2 per egg.

First, let's calculate how many eggs are eaten in a day:
- Breakfast: 3 eggs
- Baking muffins: 4 muffins

Total eggs eaten in a

## 12. Final table (dev and test)

In [28]:
print_table("DEV (train 1-10)", dev_results, 10)
print_table("TEST (test 1-30)", results, 30)

json.dump({"dev": dev_results, "test": results}, open("q1_results.json", "w"), indent=1)
print("\nsaved q1_results.json (q2.ipynb reads it)")


DEV (train 1-10)
method            accuracy      tokens    calls   
zero-shot         5/10 (50%)    2321      10      
four-shot         5/10 (50%)    1211      10      
self-consistency  5/10 (50%)    4591      40      
reflection        8/10 (80%)    3349      20      

TEST (test 1-30)
method            accuracy      tokens    calls   
zero-shot         10/30 (33%)   7681      30      
four-shot         10/30 (33%)   3446      30      
self-consistency  11/30 (37%)   15784     120     
reflection        10/30 (33%)   9687      60      

saved q1_results.json (q2.ipynb reads it)


## 13. Where extra inference helped or harmed (test)
Self-consistency and reflection start from the same prompt and demos as four-shot, so four-shot (1 call) is the baseline.
Read the listed questions in full with `show(...)` in section 11.

In [29]:
names = list(methods)
print(f"{'Q':<4}{'gold':<8}" + "".join(f"{m:<18}" for m in names))
for i, ex in enumerate(questions):
    row = [logs[(MODE, m)][i] for m in names]
    print(f"{i + 1:<4}{gold(ex):<8}" + "".join(f"{str(r[1]) + (' ok' if r[2] else ' x'):<18}" for r in row))

for extra in ["self-consistency", "reflection"]:
    pairs = list(zip(logs[(MODE, "four-shot")], logs[(MODE, extra)]))
    helped = [i + 1 for i, (a, b) in enumerate(pairs) if not a[2] and b[2]]
    harmed = [i + 1 for i, (a, b) in enumerate(pairs) if a[2] and not b[2]]
    print(f"\n{extra} vs four-shot:  helped on Q{helped}  harmed on Q{harmed}")

Q   gold    zero-shot         four-shot         self-consistency  reflection        
1   18      18 ok             18 ok             18 ok             18 ok             
2   3       3 ok              3 ok              3 ok              3 ok              
3   70000   -65000 x          65000 x           120000 x          170000 x          
4   540     540 ok            180 x             180 x             180 x             
5   20      420 x             2 x               40 x              0 x               
6   64      48 x              24 x              28.8 x            24 x              
7   260     260 ok            260 ok            260 ok            260 ok            
8   160     50 x              33733.33 x        8 x               140 x             
9   45      620 x             400 x             670 x             340 x             
10  460     1390 x            452 x             400 x             880 x             
11  366     366 ok            394 x             366 ok           